In [1]:
import pandas as pd

In [27]:
df.columns

Index(['vehicle_mass_kg', 'test_mass_kg', 'co2_wltp_g_km', 'fuel_type',
       'fuel_mode', 'engine_capacity_cc', 'engine_power_kw',
       'co2_real_world_g_km', 'fuel_efficiency_km_per_l'],
      dtype='object')

In [2]:
df=pd.read_csv("preprocessed_vehicle_data.csv")

In [29]:
df

,vehicle_mass_kg,test_mass_kg,co2_wltp_g_km,fuel_type,fuel_mode,engine_capacity_cc,engine_power_kw,co2_real_world_g_km,fuel_efficiency_km_per_l
0,1670.0,1782.0,125.0,petrol,H,2487.0,131.0,0.80,18.181818
1,1493.0,1576.0,135.0,petrol,M,1199.0,96.0,2.00,16.666667
2,1649.0,1814.0,131.0,petrol,H,1598.0,132.0,0.59,17.241379
3,1560.0,1640.0,118.0,petrol,H,1987.0,112.0,0.80,19.230769
4,1290.0,1427.0,141.0,petrol,M,999.0,81.0,1.84,16.129032
...,...,...,...,...,...,...,...,...,...
6528924,1808.0,2140.0,182.5,diesel,M,1997.0,106.0,1.10,13.513514
6528925,1475.0,1574.0,106.0,petrol,H,1798.0,72.0,0.60,21.276596
6528926,1475.0,1569.0,107.0,petrol,H,1798.0,72.0,0.60,21.276596
6528927,1808.0,2140.0,182.5,diesel,M,1997.0,106.0,1.10,13.513514


In [3]:
df_fs = df.sample(n=25_000, random_state=42)

X = df_fs.drop(columns=["fuel_efficiency_km_per_l"])
y = df_fs["fuel_efficiency_km_per_l"]

X_enc = pd.get_dummies(X, drop_first=True)


In [4]:
X_enc

,vehicle_mass_kg,test_mass_kg,co2_wltp_g_km,engine_capacity_cc,engine_power_kw,co2_real_world_g_km,fuel_type_petrol,fuel_mode_M
2212922,1393.0,1514.0,138.0,1498.0,110.0,1.68,True,True
140455,1575.0,1695.0,135.0,1469.0,96.0,2.20,True,False
1973553,1649.0,1756.0,126.0,1598.0,132.0,0.57,True,False
4289829,1242.0,1270.0,128.0,999.0,81.0,1.17,True,True
1146547,1502.0,1616.0,137.0,1998.0,137.0,0.80,True,False
...,...,...,...,...,...,...,...,...
4836961,1065.0,1165.0,110.0,999.0,52.0,1.28,True,False
3274016,1481.0,1598.0,144.0,1998.0,110.0,0.60,True,False
2452726,1161.0,1262.0,118.0,999.0,67.0,1.86,True,True
2152468,2075.0,2159.0,112.0,1998.0,167.0,1.68,True,True


In [5]:
X.dtypes[X.dtypes == "object"]


fuel_type    object
fuel_mode    object
dtype: object

In [6]:
X["fuel_type"] = X["fuel_type"].str.strip().str.lower()
X["fuel_mode"] = X["fuel_mode"].str.strip().str.lower()


In [7]:
X = pd.get_dummies(
    X,
    columns=["fuel_type", "fuel_mode"],
    drop_first=True
)


In [8]:
X.select_dtypes(include="object").columns


Index([], dtype='object')

In [9]:
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)


In [10]:
X.dtypes


vehicle_mass_kg        float64
test_mass_kg           float64
co2_wltp_g_km          float64
engine_capacity_cc     float64
engine_power_kw        float64
co2_real_world_g_km    float64
fuel_type_petrol         int64
fuel_mode_m              int64
dtype: object

In [11]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.0)
X_vt = vt.fit_transform(X)

X_vt = pd.DataFrame(
    X_vt,
    columns=vt.get_feature_names_out()
)


In [24]:
X_vt

,vehicle_mass_kg,test_mass_kg,co2_wltp_g_km,engine_capacity_cc,engine_power_kw,co2_real_world_g_km,fuel_type_petrol,fuel_mode_m
0,1393.0,1514.0,138.0,1498.0,110.0,1.68,1.0,1.0
1,1575.0,1695.0,135.0,1469.0,96.0,2.20,1.0,0.0
2,1649.0,1756.0,126.0,1598.0,132.0,0.57,1.0,0.0
3,1242.0,1270.0,128.0,999.0,81.0,1.17,1.0,1.0
4,1502.0,1616.0,137.0,1998.0,137.0,0.80,1.0,0.0
...,...,...,...,...,...,...,...,...
24995,1065.0,1165.0,110.0,999.0,52.0,1.28,1.0,0.0
24996,1481.0,1598.0,144.0,1998.0,110.0,0.60,1.0,0.0
24997,1161.0,1262.0,118.0,999.0,67.0,1.86,1.0,1.0
24998,2075.0,2159.0,112.0,1998.0,167.0,1.68,1.0,1.0


In [19]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

# Feature importances
feature_importances = pd.Series(rf.feature_importances_, index=X.columns)
feature_importances = feature_importances.sort_values(ascending=False)
print(feature_importances)


co2_wltp_g_km          0.804724
fuel_type_petrol       0.117074
test_mass_kg           0.025003
vehicle_mass_kg        0.018918
co2_real_world_g_km    0.011362
engine_capacity_cc     0.010837
engine_power_kw        0.010386
fuel_mode_m            0.001698
dtype: float64


In [20]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

rfe = RFE(estimator=RandomForestRegressor(n_estimators=100), n_features_to_select=10)
X_rfe = rfe.fit_transform(X, y)

selected_features = X.columns[rfe.support_]
print(selected_features)


C:\Anaconda3\envs\aimj\Lib\site-packages\sklearn\feature_selection\_rfe.py:300: UserWarning: Found n_features_to_select=10 > n_features=8. There will be no feature selection and all features will be kept.
  warnings.warn(


Index(['vehicle_mass_kg', 'test_mass_kg', 'co2_wltp_g_km',
       'engine_capacity_cc', 'engine_power_kw', 'co2_real_world_g_km',
       'fuel_type_petrol', 'fuel_mode_m'],
      dtype='object')


In [21]:
from sklearn.inspection import permutation_importance

result = permutation_importance(rf, X_vt, y, n_repeats=10, random_state=42)
perm_importances = pd.Series(result.importances_mean, index=X_vt.columns)
perm_importances = perm_importances.sort_values(ascending=False)
print(perm_importances)


co2_wltp_g_km          1.629904
fuel_type_petrol       0.455742
test_mass_kg           0.145250
vehicle_mass_kg        0.137535
engine_power_kw        0.076998
co2_real_world_g_km    0.074420
engine_capacity_cc     0.057905
fuel_mode_m            0.003308
dtype: float64


In [22]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_vt, y)
xgb_importances = pd.Series(xgb.feature_importances_, index=X_vt.columns).sort_values(ascending=False)
print(xgb_importances)


fuel_type_petrol       0.635535
co2_wltp_g_km          0.312916
fuel_mode_m            0.014394
engine_capacity_cc     0.008621
engine_power_kw        0.008198
co2_real_world_g_km    0.007006
test_mass_kg           0.006954
vehicle_mass_kg        0.006376
dtype: float32


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Features including CO2 (leaky)
X_leaky = X_vt.copy()  # X_vt has 'co2_wltp_g_km' + other features

# Features without CO2 (realistic)
X_real = X_vt.drop(columns=['co2_wltp_g_km', 'co2_real_world_g_km'])

# Target
y_target = y



In [13]:
X_real

,vehicle_mass_kg,test_mass_kg,engine_capacity_cc,engine_power_kw,fuel_type_petrol,fuel_mode_m
0,1393.0,1514.0,1498.0,110.0,1.0,1.0
1,1575.0,1695.0,1469.0,96.0,1.0,0.0
2,1649.0,1756.0,1598.0,132.0,1.0,0.0
3,1242.0,1270.0,999.0,81.0,1.0,1.0
4,1502.0,1616.0,1998.0,137.0,1.0,0.0
...,...,...,...,...,...,...
24995,1065.0,1165.0,999.0,52.0,1.0,0.0
24996,1481.0,1598.0,1998.0,110.0,1.0,0.0
24997,1161.0,1262.0,999.0,67.0,1.0,1.0
24998,2075.0,2159.0,1998.0,167.0,1.0,1.0


In [14]:
X_train_leaky, X_test_leaky, y_train, y_test = train_test_split(X_leaky, y_target, test_size=0.2, random_state=42)
X_train_real, X_test_real, _, _ = train_test_split(X_real, y_target, test_size=0.2, random_state=42)


In [15]:
rf_leaky = RandomForestRegressor(n_estimators=100, random_state=42)
rf_leaky.fit(X_train_leaky, y_train)
y_pred_leaky = rf_leaky.predict(X_test_leaky)

r2_leaky = r2_score(y_test, y_pred_leaky)
print("R² with CO2 (leaky):", r2_leaky)


R² with CO2 (leaky): 0.9506778640035451


In [16]:
rf_real = RandomForestRegressor(n_estimators=100, random_state=42)
rf_real.fit(X_train_real, y_train)
y_pred_real = rf_real.predict(X_test_real)

r2_real = r2_score(y_test, y_pred_real)
print("R² without CO2 (realistic):", r2_real)


R² without CO2 (realistic): 0.9217038975249304


In [19]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

# X_final = your features without CO2 columns
# y_target = your target (fuel consumption)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_test_real, y_test)

# Get feature importances
feat_importances = pd.Series(rf.feature_importances_, index=X_test_real.columns)
feat_importances = feat_importances.sort_values(ascending=False)

print("Random Forest feature importances (without CO2):\n", feat_importances)


Random Forest feature importances (without CO2):
 engine_power_kw       0.444061
vehicle_mass_kg       0.162382
test_mass_kg          0.157186
engine_capacity_cc    0.098835
fuel_type_petrol      0.086347
fuel_mode_m           0.051190
dtype: float64


In [22]:
# Remove low-importance feature
X_train_real = X_train_real.drop(columns=['fuel_mode_m'])


In [23]:
X_train_real

,vehicle_mass_kg,test_mass_kg,engine_capacity_cc,engine_power_kw,fuel_type_petrol
23311,1545.0,1648.0,1998.0,131.0,1.0
23623,2133.5,2287.0,3124.5,167.0,1.0
1020,1592.0,1748.0,1199.0,96.0,1.0
12645,1688.0,1820.0,1969.0,145.0,1.0
1533,1262.0,1367.0,999.0,84.0,1.0
...,...,...,...,...,...
21575,1436.0,1526.0,1499.0,96.0,0.0
5390,1365.0,1463.0,999.0,88.0,1.0
860,1975.0,2191.0,2967.0,167.0,0.0
15795,1567.0,1667.0,1499.0,96.0,0.0


In [ ]:
# Drop the column from your features
df_final = df.drop(columns=[
    'fuel_mode',
    'co2_wltp_g_km',
    'co2_real_world_g_km'
])

In [30]:
df_final

,vehicle_mass_kg,test_mass_kg,fuel_type,engine_capacity_cc,engine_power_kw,fuel_efficiency_km_per_l
0,1670.0,1782.0,petrol,2487.0,131.0,18.181818
1,1493.0,1576.0,petrol,1199.0,96.0,16.666667
2,1649.0,1814.0,petrol,1598.0,132.0,17.241379
3,1560.0,1640.0,petrol,1987.0,112.0,19.230769
4,1290.0,1427.0,petrol,999.0,81.0,16.129032
...,...,...,...,...,...,...
6528924,1808.0,2140.0,diesel,1997.0,106.0,13.513514
6528925,1475.0,1574.0,petrol,1798.0,72.0,21.276596
6528926,1475.0,1569.0,petrol,1798.0,72.0,21.276596
6528927,1808.0,2140.0,diesel,1997.0,106.0,13.513514


In [31]:
df_final.to_csv("feature_selected_data.csv", index=False)
